In [ ]:
###################################################################################################
# 參數設定
audio_file = "test_audio.mp3"  # 你的本地錄音檔案名稱(含副檔名)
lang = "auto"                  # "auto" 或 "zh"/"en"/"ja"/"ko"
output_format = "txt"          # 輸出格式 "txt" 或 "srt"
model_id = "MediaTek-Research/Breeze-ASR-26"
output_path = "."              # 輸出資料夾，預設為目前目錄
chunk_length_s = 30             # 長音訊可用分段推論；設為 0 停用
stride_length_s = 5             # 分段交疊秒數

overwrite = False               # 是否覆蓋已存在的辨識結果 (True or False)
verbose = False                 # 是否顯示完整推論回傳 (True or False)
####################################################################################################

import os
import torch
import torchaudio
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    AutomaticSpeechRecognitionPipeline,
)

# 檢查檔案是否存在
if not os.path.exists(audio_file):
    print(f"找不到音訊檔案: {audio_file}")
    raise SystemExit(1)

# 裝置設定
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE_INDEX = 0 if DEVICE == "cuda" else -1
TORCH_DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

# 語言設定
lang_normalized = str(lang).lower().strip()
if lang_normalized in ["auto", "none", "", "自動判斷"]:
    language_name = None
elif lang_normalized in ["zh", "chinese", "中文", "zh-tw", "zh_tw"]:
    language_name = "chinese"
elif lang_normalized in ["en", "english", "英文"]:
    language_name = "english"
elif lang_normalized in ["ja", "japanese", "日文"]:
    language_name = "japanese"
elif lang_normalized in ["ko", "korean", "韓文"]:
    language_name = "korean"
else:
    language_name = lang_normalized

# 載入音訊並統一為 16kHz 單聲道
waveform, sample_rate = torchaudio.load(audio_file)
if waveform.dim() > 1:
    waveform = waveform.mean(dim=0)
if sample_rate != 16_000:
    resampler = torchaudio.transforms.Resample(sample_rate, 16_000)
    waveform = resampler(waveform)
    sample_rate = 16_000
waveform = waveform.squeeze().numpy()
audio_duration_s = len(waveform) / sample_rate if sample_rate > 0 else 0

# 載入模型
processor = WhisperProcessor.from_pretrained(model_id)
model = WhisperForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=TORCH_DTYPE,
    low_cpu_mem_usage=True,
).to(DEVICE)
model.eval()

# 建立 ASR pipeline
asr_pipeline = AutomaticSpeechRecognitionPipeline(
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    chunk_length_s=chunk_length_s,
    stride_length_s=stride_length_s,
    device=DEVICE_INDEX,
)

generate_kwargs = {"task": "transcribe"}
if language_name is not None:
    generate_kwargs["language"] = language_name

return_timestamps = output_format == "srt"

# 推論
result = asr_pipeline(
    {"array": waveform, "sampling_rate": sample_rate},
    return_timestamps=return_timestamps,
    generate_kwargs=generate_kwargs,
)

if verbose:
    print(result)

# 設定輸出檔名
base_name = os.path.splitext(os.path.basename(audio_file))[0]
output_file = os.path.join(output_path, f"{base_name}.{output_format}")

# 檢查是否需要覆蓋
count = 0
while os.path.exists(output_file) and not overwrite:
    count += 1
    output_file = os.path.join(output_path, f"{base_name}_{count}.{output_format}")


def _format_timestamp(seconds: float) -> str:
    millis = int(max(seconds, 0) * 1000)
    hours = millis // 3_600_000
    minutes = (millis % 3_600_000) // 60_000
    secs = (millis % 60_000) // 1000
    ms = millis % 1000
    return f"{hours:02d}:{minutes:02d}:{secs:02d},{ms:03d}"


def _extract_segments(asr_output: dict) -> list[dict]:
    chunks = asr_output.get("chunks") or []
    segments = []
    for chunk in chunks:
        ts = chunk.get("timestamp") or chunk.get("timestamps")
        if not ts or len(ts) != 2:
            continue
        start, end = ts
        if start is None or end is None:
            continue
        text = (chunk.get("text") or "").strip()
        if not text:
            continue
        segments.append({"start": float(start), "end": float(end), "text": text})
    return segments


def _write_srt(segments: list[dict], path: str) -> None:
    with open(path, "w", encoding="utf-8") as f:
        for idx, seg in enumerate(segments, 1):
            f.write(f"{idx}\n")
            f.write(
                f"{_format_timestamp(seg['start'])} --> { _format_timestamp(seg['end']) }\n"
            )
            f.write(f"{seg['text']}\n\n")


if output_format == "txt":
    with open(output_file, "w", encoding="utf-8") as f:
        f.write(result.get("text", ""))
elif output_format == "srt":
    segments = _extract_segments(result)
    if not segments and result.get("text"):
        segments = [
            {"start": 0.0, "end": float(audio_duration_s), "text": result["text"].strip()}
        ]
    _write_srt(segments, output_file)
else:
    raise ValueError("output_format 只支援 'txt' 或 'srt'")

print(f"辨識完成，結果已存至 {output_file}")


無可用硬體加速，使用 CPU 執行


C:\Users\chiuj\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
100%|██████████| 4051/4051 [01:15<00:00, 53.46frames/s]

辨識完成，結果已存至 .\test_audio.txt
